<a href="https://colab.research.google.com/github/Kenny625819/Applied-Data-Science/blob/main/IEEE_reanalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
ESCC (Bilsky scale) Inter-Rater Agreement and Consensus Label Generation
[177-case version]

This script performs the following steps:

1. Load ESCC ratings from an Excel file (three raters per case).
2. Compute overall inter-rater agreement using Fleiss' kappa.
3. Compute pairwise Cohen's kappa (unweighted and quadratic-weighted)
   for each rater pair.
4. Generate a consensus ESCC label for each case:
   - If at least two raters agree, the majority label is used.
   - If all three raters disagree, the median label is chosen based on
     the ordinal Bilsky scale.
5. Save the original ratings and the consensus label to a new Excel file.

Expected input format
---------------------
The input Excel file must contain the following columns:

    ESCC_1  : ESCC label from rater 1 (e.g., "1a", "1b", "1c", "2", "3")
    ESCC_2  : ESCC label from rater 2
    ESCC_3  : ESCC label from rater 3

The Bilsky/ESCC labels are treated as an ordinal scale with the following order:
    1a < 1b < 1c < 2 < 3
"""

import numpy as np
import pandas as pd
from collections import Counter
from sklearn.metrics import cohen_kappa_score


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
INPUT_EXCEL = "ESCC_3_177.xlsx"
OUTPUT_EXCEL = "ESCC_3_with_consensus_177.xlsx"

# Bilsky/ESCC ordinal scale
ORDERED_LABELS = ["1a", "1b", "1c", "2", "3"]

LABEL_TO_IDX = {lab: i for i, lab in enumerate(ORDERED_LABELS)}
IDX_TO_LABEL = {i: lab for lab, i in LABEL_TO_IDX.items()}


# ---------------------------------------------------------------------
# Fleiss' kappa
# ---------------------------------------------------------------------
def fleiss_kappa(M: np.ndarray):
    """
    Compute Fleiss' kappa for multiple raters.

    Parameters
    ----------
    M : np.ndarray of shape (N, k)
        M[i, j] is the number of raters who assigned category j to case i.

    Returns
    -------
    kappa : float
        Fleiss' kappa.
    P_bar : float
        Mean observed agreement.
    P_e : float
        Expected agreement by chance.
    """
    N, k = M.shape
    n = M.sum(axis=1)[0]  # number of raters per case (assumed constant)

    p_j = M.sum(axis=0) / (N * n)
    P_i = ((M * (M - 1)).sum(axis=1)) / (n * (n - 1))

    P_bar = P_i.mean()
    P_e = (p_j ** 2).sum()

    kappa = (P_bar - P_e) / (1 - P_e)
    return kappa, P_bar, P_e


# ---------------------------------------------------------------------
# Quadratic-weighted Cohen's kappa
# ---------------------------------------------------------------------
def quadratic_weighted_kappa(a, b, labels_order):
    """
    Compute quadratic-weighted Cohen's kappa between two raters.
    """
    lab_to_i = {lab: i for i, lab in enumerate(labels_order)}
    a_idx = np.array([lab_to_i[str(x).strip()] for x in a])
    b_idx = np.array([lab_to_i[str(x).strip()] for x in b])
    n_cat = len(labels_order)

    # Weight matrix
    w = np.zeros((n_cat, n_cat))
    for i in range(n_cat):
        for j in range(n_cat):
            w[i, j] = ((i - j) ** 2) / ((n_cat - 1) ** 2)

    # Observed matrix
    O = np.zeros((n_cat, n_cat))
    for x, y in zip(a_idx, b_idx):
        O[x, y] += 1
    O = O / O.sum()

    # Expected matrix
    a_hist = np.bincount(a_idx, minlength=n_cat)
    b_hist = np.bincount(b_idx, minlength=n_cat)
    E = np.outer(a_hist, b_hist)
    E = E / E.sum()

    num = (w * O).sum()
    den = (w * E).sum()
    return 1.0 - num / den


# ---------------------------------------------------------------------
# Consensus label generation
# ---------------------------------------------------------------------
def consensus_row(row, raters, label_to_idx, idx_to_label):
    """
    Generate a consensus ESCC label for a single case.

    Rules:
    - If at least two raters agree, return the majority label.
    - If all three raters disagree, return the median label on the
      ordinal Bilsky scale (1a < 1b < 1c < 2 < 3).
    """
    vals = [str(row[x]).strip() for x in raters]
    counts = Counter(vals)
    most_common_label, freq = counts.most_common(1)[0]

    if freq >= 2:
        return most_common_label

    idxs = sorted(label_to_idx[v] for v in vals)
    median_idx = idxs[1]
    return idx_to_label[median_idx]


# ---------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------
def main():
    print(f"Loading input file: {INPUT_EXCEL}")
    df = pd.read_excel(INPUT_EXCEL)
    print("Input columns:", list(df.columns))
    print(f"Number of cases: {len(df)}")

    raters = ["ESCC_1", "ESCC_2", "ESCC_3"]

    # Check required columns
    missing_cols = [c for c in raters if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    # Standardize labels as strings
    for c in raters:
        df[c] = df[c].astype(str).str.strip()

    # Build N x k matrix for Fleiss' kappa
    N = len(df)
    k = len(ORDERED_LABELS)
    M = np.zeros((N, k), dtype=int)

    for i, row in df[raters].iterrows():
        for val in row:
            if pd.isna(val):
                continue
            if val not in LABEL_TO_IDX:
                raise ValueError(f"Unexpected ESCC label found: {val}")
            M[i, LABEL_TO_IDX[val]] += 1

    # Fleiss' kappa
    fleiss_k, P_bar, P_e = fleiss_kappa(M)
    print("\n=== Fleiss' kappa ===")
    print(f"Fleiss' kappa: {fleiss_k:.3f}")
    print(f"P_bar (mean agreement): {P_bar:.3f}")
    print(f"P_e (chance agreement): {P_e:.3f}")

    # Pairwise Cohen's kappa
    print("\n=== Pairwise Cohen's kappa ===")
    pairs = [("ESCC_1", "ESCC_2"), ("ESCC_1", "ESCC_3"), ("ESCC_2", "ESCC_3")]

    for c1, c2 in pairs:
        a = df[c1].values
        b = df[c2].values

        k_unw = cohen_kappa_score(a, b)
        k_w = quadratic_weighted_kappa(a, b, ORDERED_LABELS)

        print(f"{c1} vs {c2}: Unweighted = {k_unw:.3f}, Quadratic-weighted = {k_w:.3f}")

    # Consensus labels
    df["ESCC_consensus"] = df.apply(
        consensus_row,
        axis=1,
        args=(raters, LABEL_TO_IDX, IDX_TO_LABEL),
    )

    print("\n=== Consensus ESCC distribution ===")
    print(df["ESCC_consensus"].value_counts().sort_index())

    full_disagree = (
        (df["ESCC_1"] != df["ESCC_2"]) &
        (df["ESCC_1"] != df["ESCC_3"]) &
        (df["ESCC_2"] != df["ESCC_3"])
    ).sum()

    print(f"\nNumber of full-disagreement cases: {full_disagree}")

    # Save output
    df.to_excel(OUTPUT_EXCEL, index=False)
    print(f"\nSaved consensus results to: {OUTPUT_EXCEL}")


if __name__ == "__main__":
    main()


Loading input file: ESCC_3_177.xlsx
Input columns: ['filename', 'ESCC_1', 'ESCC_2', 'ESCC_3']
Number of cases: 177

=== Fleiss' kappa ===
Fleiss' kappa: 0.579
P_bar (mean agreement): 0.750
P_e (chance agreement): 0.405

=== Pairwise Cohen's kappa ===
ESCC_1 vs ESCC_2: Unweighted = 0.684, Quadratic-weighted = 0.887
ESCC_1 vs ESCC_3: Unweighted = 0.510, Quadratic-weighted = 0.787
ESCC_2 vs ESCC_3: Unweighted = 0.550, Quadratic-weighted = 0.804

=== Consensus ESCC distribution ===
ESCC_consensus
1b     20
1c      8
2      49
3     100
Name: count, dtype: int64

Number of full-disagreement cases: 5

Saved consensus results to: ESCC_3_with_consensus_177.xlsx


In [2]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Dataset Construction for CNN-based ESCC Classification
[177-case version | submission-ready]

This script performs the following steps:
1. Detects the working base directory automatically (/content or /mnt/data).
2. Unzips the MRI image archive if needed.
3. Loads consensus ESCC labels from Excel.
4. Scans the image directory and maps numeric IDs to real image filenames.
5. Matches each case in the Excel file to its image.
6. Generates dataset_177.csv for CNN training.

Expected input files:
    - ESCC_3_with_consensus_177.xlsx
    - images177.zip
      or extracted folder:
    - images177/

Output:
    - dataset_177.csv

Required Excel columns:
    - filename
    - ESCC_consensus
"""

import os
import zipfile
import pandas as pd


# ---------------------------------------------------------------------
# Utility functions
# ---------------------------------------------------------------------
def detect_base_dir():
    """
    Automatically detect the working base directory.

    Priority:
    1. /content   (Google Colab standard)
    2. /mnt/data  (some notebook/container environments)
    """
    candidates = ["/content", "/mnt/data"]

    for base in candidates:
        if os.path.exists(base):
            return base

    raise FileNotFoundError(
        "No valid base directory found. Neither '/content' nor '/mnt/data' exists."
    )


def resolve_existing_path(*paths):
    """
    Return the first existing path among candidates.
    If none exists, return the first candidate for default output use.
    """
    for p in paths:
        if os.path.exists(p):
            return p
    return paths[0]


def unzip_if_needed(zip_file, image_dir, extract_to):
    """
    Unzip image archive if the extracted folder does not exist.
    """
    if os.path.exists(image_dir) and os.path.isdir(image_dir):
        print(f"[INFO] Image directory already exists: {image_dir}")
        return

    if not os.path.exists(zip_file):
        raise FileNotFoundError(
            f"ZIP file not found: {zip_file}\n"
            f"Please confirm that 'images177.zip' has been uploaded."
        )

    print(f"[INFO] Unzipping: {zip_file}")
    with zipfile.ZipFile(zip_file, "r") as zf:
        zf.extractall(extract_to)
    print("[INFO] Unzip completed.")


def build_file_map(image_dir):
    """
    Build mapping from numeric file stem to actual filename.

    Example:
        '12.png' -> file_map['12'] = '12.png'
    """
    if not os.path.exists(image_dir):
        raise FileNotFoundError(f"Image directory not found: {image_dir}")

    img_files = os.listdir(image_dir)
    file_map = {}

    for f in img_files:
        full_path = os.path.join(image_dir, f)
        if not os.path.isfile(full_path):
            continue

        name, ext = os.path.splitext(f)
        if name.isdigit():
            file_map[name] = f

    return file_map


def normalize_filename_key(raw_value):
    """
    Normalize Excel filename values such as:
        12, 12.0, '12', '12.png'
    into:
        '12'
    """
    if pd.isna(raw_value):
        return None

    s = str(raw_value).strip()

    # Remove extension if present
    s_no_ext = os.path.splitext(s)[0]

    # Try numeric normalization first
    try:
        return str(int(float(s_no_ext)))
    except Exception:
        return s_no_ext.strip()


# ---------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------
def main():
    base_dir = detect_base_dir()
    print(f"[INFO] Detected base directory: {base_dir}")

    # Input/output paths
    zip_file = resolve_existing_path(
        os.path.join("/content", "images177.zip"),
        os.path.join("/mnt/data", "images177.zip"),
    )

    excel_path = resolve_existing_path(
        os.path.join("/content", "ESCC_3_with_consensus_177.xlsx"),
        os.path.join("/mnt/data", "ESCC_3_with_consensus_177.xlsx"),
    )

    image_dir = resolve_existing_path(
        os.path.join("/content", "images177"),
        os.path.join("/mnt/data", "images177"),
    )

    output_csv = os.path.join(base_dir, "dataset_177.csv")

    print("\n=== Path Configuration ===")
    print(f"ZIP_FILE   : {zip_file}")
    print(f"IMAGE_DIR  : {image_dir}")
    print(f"EXCEL_PATH : {excel_path}")
    print(f"OUTPUT_CSV : {output_csv}")

    # 1. Unzip if needed
    unzip_if_needed(zip_file, image_dir, base_dir)

    # Re-resolve image_dir in case it was created by unzip
    image_dir = resolve_existing_path(
        os.path.join("/content", "images177"),
        os.path.join("/mnt/data", "images177"),
    )

    # 2. Check image directory
    print("\n=== Directory Check ===")
    print(f"Image directory: {image_dir}")

    if not os.path.exists(image_dir):
        raise FileNotFoundError(f"Image directory not found after unzip: {image_dir}")

    image_list = os.listdir(image_dir)
    print(f"Total files in image directory: {len(image_list)}")
    print(f"First 10 files: {image_list[:10]}")

    # 3. Load consensus labels
    if not os.path.exists(excel_path):
        raise FileNotFoundError(
            f"Excel file not found: {excel_path}\n"
            f"Please confirm that 'ESCC_3_with_consensus_177.xlsx' exists."
        )

    print(f"\n[INFO] Loading label file: {excel_path}")
    df = pd.read_excel(excel_path)

    print(f"Number of rows in Excel: {len(df)}")
    print(f"Columns: {list(df.columns)}")

    required_cols = ["filename", "ESCC_consensus"]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns in Excel: {missing_cols}")

    # 4. Build image filename mapping
    file_map = build_file_map(image_dir)
    print(f"Mapped numeric image IDs: {len(file_map)}")

    # 5. Match Excel rows to images
    rows = []
    missing = []

    for _, row in df.iterrows():
        key = normalize_filename_key(row["filename"])

        if key is None:
            missing.append(row["filename"])
            continue

        if key in file_map:
            rows.append({
                "filename": file_map[key],
                "label": str(row["ESCC_consensus"]).strip()
            })
        else:
            missing.append(key)

    # 6. Save dataset
    dataset = pd.DataFrame(rows)

    if dataset.empty:
        raise ValueError(
            "No images were matched. Please check whether the Excel 'filename' column "
            "corresponds to the numeric image file names in the extracted folder."
        )

    dataset.to_csv(output_csv, index=False)

    # 7. Summary
    print("\n=== Dataset Summary ===")
    print(f"Total mapped images: {len(dataset)}")

    if len(missing) > 0:
        print(f"Missing files: {len(missing)}")
        print(f"Example missing IDs: {missing[:10]}")
    else:
        print("All images successfully mapped.")

    print(f"\nSaved dataset file: {output_csv}")

    print("\n=== Label Distribution ===")
    print(dataset["label"].value_counts(dropna=False).sort_index())

    print("\n[INFO] Dataset construction completed successfully.")


if __name__ == "__main__":
    main()


[INFO] Detected base directory: /content

=== Path Configuration ===
ZIP_FILE   : /content/images177.zip
IMAGE_DIR  : /content/images177
EXCEL_PATH : /content/ESCC_3_with_consensus_177.xlsx
OUTPUT_CSV : /content/dataset_177.csv
[INFO] Unzipping: /content/images177.zip
[INFO] Unzip completed.

=== Directory Check ===
Image directory: /content/images177
Total files in image directory: 177
First 10 files: ['61.PNG', '139.png', '143.png', '89.PNG', '90.PNG', '12.png', '212.PNG', '46.PNG', '88.PNG', '97.PNG']

[INFO] Loading label file: /content/ESCC_3_with_consensus_177.xlsx
Number of rows in Excel: 177
Columns: ['filename', 'ESCC_1', 'ESCC_2', 'ESCC_3', 'ESCC_consensus']
Mapped numeric image IDs: 177

=== Dataset Summary ===
Total mapped images: 177
All images successfully mapped.

Saved dataset file: /content/dataset_177.csv

=== Label Distribution ===
label
1b     20
1c      8
2      49
3     100
Name: count, dtype: int64

[INFO] Dataset construction completed successfully.


In [3]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
OOF CNN Training for ESCC Classification
[177-case version | local/Jupyter compatible]

This script performs:
1. Checks dataset_177.csv and image folder
2. Extracts images177.zip if needed
3. Four-class ESCC classification using ResNet-18
4. Five-fold stratified out-of-fold (OOF) cross-validation
5. Fold-wise model checkpoint saving
6. Export of OOF predictions for downstream multimodal survival modeling

Input:
    - dataset_177.csv
    - images177/   or   images177.zip

Output:
    - models_escc_resnet18_177/
        - resnet18_escc_fold1.pth ...
        - cv_results.csv
    - escc_oof_predictions_177.csv
"""

# =============================================================================
# Imports
# =============================================================================
import os
import random
import zipfile
import copy
import numpy as np
import pandas as pd
from PIL import Image

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms


# =============================================================================
# Configuration
# =============================================================================
DATASET_CSV = "dataset_177.csv"
ZIP_FILE = "images177.zip"
IMAGE_DIR = "images177"
OUTPUT_DIR = "models_escc_resnet18_177"

NUM_EPOCHS = 50
BATCH_SIZE = 16
LR = 1e-4
PATIENCE = 10
NUM_FOLDS = 5
SEED = 42
USE_PRETRAINED = True
NUM_WORKERS = 0   # Jupyter/Windows/Colabで安定しやすい

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Using device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# =============================================================================
# Reproducibility
# =============================================================================
def set_seed(seed=42):
    """Fix random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)


# =============================================================================
# Utilities
# =============================================================================
def ensure_image_directory(zip_file, image_dir):
    """
    Ensure image directory exists.
    If image_dir does not exist but zip_file exists, unzip it.

    Also supports:
        - images177/*.png
        - images177/images177/*.png
    Returns the resolved directory that actually contains image files.
    """
    if not os.path.isdir(image_dir):
        if os.path.isfile(zip_file):
            print(f"Extracting archive: {zip_file}")
            with zipfile.ZipFile(zip_file, "r") as zf:
                zf.extractall(".")
            print("Extraction completed.")
        else:
            raise FileNotFoundError(
                f"Neither image folder nor zip file was found:\n"
                f" - folder: {image_dir}\n"
                f" - zip   : {zip_file}"
            )

    if not os.path.isdir(image_dir):
        raise FileNotFoundError(f"Image directory not found after extraction: {image_dir}")

    # Case 1: image files are directly inside image_dir
    direct_files = [
        f for f in os.listdir(image_dir)
        if os.path.isfile(os.path.join(image_dir, f))
    ]
    direct_images = [
        f for f in direct_files
        if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"))
    ]
    if len(direct_images) > 0:
        print(f"Resolved image directory: {image_dir}")
        print(f"Number of image files: {len(direct_images)}")
        print("Sample files:", sorted(direct_images)[:10])
        return image_dir

    # Case 2: nested subfolder
    subdirs = [
        d for d in os.listdir(image_dir)
        if os.path.isdir(os.path.join(image_dir, d))
    ]
    for sub in subdirs:
        nested_dir = os.path.join(image_dir, sub)
        nested_files = [
            f for f in os.listdir(nested_dir)
            if os.path.isfile(os.path.join(nested_dir, f))
        ]
        nested_images = [
            f for f in nested_files
            if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"))
        ]
        if len(nested_images) > 0:
            print(f"Resolved nested image directory: {nested_dir}")
            print(f"Number of image files: {len(nested_images)}")
            print("Sample files:", sorted(nested_images)[:10])
            return nested_dir

    raise FileNotFoundError(
        f"No image files were found in '{image_dir}' or its immediate subdirectories."
    )


def compute_auc(all_idx, probs, all_str, label_to_idx):
    """
    Compute:
      1) multi-class macro AUC
      2) binary AUC for high-grade ESCC (2/3 vs 1b/1c)
    """
    # multi-class macro AUC
    try:
        multi_auc = roc_auc_score(all_idx, probs, multi_class="ovr", average="macro")
    except Exception:
        multi_auc = np.nan

    # binary high-grade AUC
    high_grade = {"2", "3"}
    y_bin = np.array([1 if s in high_grade else 0 for s in all_str], dtype=int)

    present_high = [lab for lab in ["2", "3"] if lab in label_to_idx]
    if len(present_high) == 0:
        binary_auc = np.nan
    else:
        high_idx = [label_to_idx[h] for h in present_high]
        prob_high = probs[:, high_idx].sum(axis=1)
        try:
            binary_auc = roc_auc_score(y_bin, prob_high)
        except Exception:
            binary_auc = np.nan

    return multi_auc, binary_auc


def get_class_weights(labels, ncls):
    """Inverse-frequency class weights."""
    counts = np.bincount(labels, minlength=ncls)
    total = counts.sum()
    w = total / (counts + 1e-6)
    w = w / w.sum()
    return torch.tensor(w, dtype=torch.float32)


def create_model(num_classes, pretrained=True):
    """Create ResNet-18 classifier."""
    try:
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        model = models.resnet18(weights=weights)
    except Exception:
        model = models.resnet18(pretrained=pretrained)

    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model


# =============================================================================
# Resolve image directory
# =============================================================================
IMG_DIR = ensure_image_directory(ZIP_FILE, IMAGE_DIR)


# =============================================================================
# Load dataset
# =============================================================================
if not os.path.isfile(DATASET_CSV):
    raise FileNotFoundError(f"Dataset CSV not found: {DATASET_CSV}")

df = pd.read_csv(DATASET_CSV)
df["label"] = df["label"].astype(str).str.strip()
df["filename"] = df["filename"].astype(str).str.strip()

print("\n=== Dataset Check ===")
print("Number of cases:", len(df))
print(df.head())

required_cols = ["filename", "label"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in dataset CSV: {missing_cols}")

unique_labels = sorted(df["label"].unique())
label_to_idx = {lab: i for i, lab in enumerate(unique_labels)}
idx_to_label = {i: lab for lab, i in label_to_idx.items()}
df["label_idx"] = df["label"].map(label_to_idx)

num_classes = len(unique_labels)
print("Label mapping:", label_to_idx)

# verify image files
missing_files = [
    f for f in df["filename"]
    if not os.path.isfile(os.path.join(IMG_DIR, f))
]
if len(missing_files) > 0:
    print("\nMissing image files detected:")
    print(missing_files[:20])
    raise FileNotFoundError(
        f"{len(missing_files)} image files listed in {DATASET_CSV} were not found in {IMG_DIR}."
    )
else:
    print("All image files listed in dataset_177.csv were found.")


# =============================================================================
# Dataset class
# =============================================================================
class ESCCDataset(Dataset):
    def __init__(self, df_in, img_dir, transform=None):
        self.df = df_in.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.loc[idx]
        img_name = row["filename"]
        img_path = os.path.join(self.img_dir, img_name)

        image = Image.open(img_path).convert("RGB")
        label_idx = int(row["label_idx"])
        label_str = row["label"]

        if self.transform:
            image = self.transform(image)

        return image, label_idx, label_str, img_name


# =============================================================================
# Image transforms
# =============================================================================
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomAffine(degrees=10, translate=(0.05, 0.05)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])


# =============================================================================
# Train / eval loops
# =============================================================================
def train_one_epoch(model, loader, criterion, optimizer, label_to_idx):
    model.train()
    total_loss = 0.0
    all_idx, all_str, probs_list = [], [], []

    for x, y_idx, y_str, _ in loader:
        x = x.to(DEVICE)
        y_idx = y_idx.to(DEVICE)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y_idx)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        probs = F.softmax(out, dim=1).detach().cpu().numpy()

        probs_list.append(probs)
        all_idx.append(y_idx.cpu().numpy())
        all_str.extend(list(y_str))

    probs = np.concatenate(probs_list, axis=0)
    all_idx = np.concatenate(all_idx, axis=0)
    multi_auc, binary_auc = compute_auc(all_idx, probs, np.array(all_str), label_to_idx)

    return total_loss / len(loader.dataset), multi_auc, binary_auc


def eval_one_epoch(model, loader, criterion, label_to_idx, return_predictions=False):
    model.eval()
    total_loss = 0.0
    all_idx, all_str, probs_list, filenames = [], [], [], []

    with torch.no_grad():
        for x, y_idx, y_str, img_name in loader:
            x = x.to(DEVICE)
            y_idx = y_idx.to(DEVICE)

            out = model(x)
            loss = criterion(out, y_idx)
            total_loss += loss.item() * x.size(0)

            probs = F.softmax(out, dim=1).cpu().numpy()

            probs_list.append(probs)
            all_idx.append(y_idx.cpu().numpy())
            all_str.extend(list(y_str))
            filenames.extend(list(img_name))

    probs = np.concatenate(probs_list, axis=0)
    all_idx = np.concatenate(all_idx, axis=0)
    multi_auc, binary_auc = compute_auc(all_idx, probs, np.array(all_str), label_to_idx)

    if return_predictions:
        return total_loss / len(loader.dataset), multi_auc, binary_auc, probs, all_idx, all_str, filenames
    return total_loss / len(loader.dataset), multi_auc, binary_auc


# =============================================================================
# 5-fold Stratified OOF CV
# =============================================================================
skf = StratifiedKFold(
    n_splits=NUM_FOLDS,
    shuffle=True,
    random_state=SEED
)

fold_results = []
oof_records = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(df["filename"], df["label_idx"])):
    print(f"\n===== Fold {fold + 1}/{NUM_FOLDS} =====")

    df_train = df.iloc[tr_idx].copy()
    df_val = df.iloc[va_idx].copy()

    train_ds = ESCCDataset(df_train, IMG_DIR, train_transform)
    val_ds = ESCCDataset(df_val, IMG_DIR, val_transform)

    train_dl = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS
    )
    val_dl = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS
    )

    class_weights = get_class_weights(df_train["label_idx"].values, num_classes).to(DEVICE)

    model = create_model(num_classes, pretrained=USE_PRETRAINED).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    best_auc = -np.inf
    best_state = None
    best_val_binary_auc = np.nan
    wait = 0

    for ep in range(1, NUM_EPOCHS + 1):
        tr_loss, tr_mauc, tr_bauc = train_one_epoch(
            model, train_dl, criterion, optimizer, label_to_idx
        )
        va_loss, va_mauc, va_bauc = eval_one_epoch(
            model, val_dl, criterion, label_to_idx
        )

        print(
            f"[Ep {ep}] "
            f"Train Loss {tr_loss:.4f} | mAUC {tr_mauc:.4f} | bAUC {tr_bauc:.4f} || "
            f"Val Loss {va_loss:.4f} | mAUC {va_mauc:.4f} | bAUC {va_bauc:.4f}"
        )

        # early stopping based on validation macro-AUC
        if np.isnan(va_mauc):
            wait += 1
        elif va_mauc > best_auc + 1e-4:
            best_auc = va_mauc
            best_val_binary_auc = va_bauc
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

        if wait >= PATIENCE:
            print("Early stopping.")
            break

    if best_state is None:
        best_state = copy.deepcopy(model.state_dict())

    model_path = f"resnet18_escc_fold{fold + 1}.pth"
    torch.save(best_state, os.path.join(OUTPUT_DIR, model_path))

    # load best model and generate OOF predictions for validation fold
    model.load_state_dict(best_state)
    _, final_mauc, final_bauc, probs, true_idx, true_str, filenames = eval_one_epoch(
        model, val_dl, criterion, label_to_idx, return_predictions=True
    )

    prob_df = pd.DataFrame(probs, columns=[f"prob_{lab}" for lab in unique_labels])
    pred_idx = probs.argmax(axis=1)
    pred_label = [idx_to_label[i] for i in pred_idx]

    present_high = [lab for lab in ["2", "3"] if lab in label_to_idx]
    high_idx = [label_to_idx[h] for h in present_high]
    high_prob = probs[:, high_idx].sum(axis=1) if len(high_idx) > 0 else np.nan

    fold_oof = pd.DataFrame({
        "filename": filenames,
        "true_label": true_str,
        "pred_label": pred_label,
        "high_prob": high_prob,
        "fold": fold + 1
    })
    fold_oof = pd.concat([fold_oof, prob_df], axis=1)
    oof_records.append(fold_oof)

    fold_results.append({
        "fold": fold + 1,
        "val_multi_auc": final_mauc,
        "val_binary_auc": final_bauc,
        "model_path": model_path
    })


# =============================================================================
# Save CV results
# =============================================================================
cv = pd.DataFrame(fold_results)
cv.to_csv(os.path.join(OUTPUT_DIR, "cv_results.csv"), index=False)

print("\n===== Cross-validation Results =====")
print(cv)
print("Mean multi-class AUC:", cv["val_multi_auc"].mean())
print("Mean binary AUC:", cv["val_binary_auc"].mean())


# =============================================================================
# Save OOF predictions
# =============================================================================
oof_df = pd.concat(oof_records, axis=0).reset_index(drop=True)
oof_df = oof_df.sort_values("filename").reset_index(drop=True)
oof_df.to_csv("escc_oof_predictions_177.csv", index=False)

print("\nSaved: escc_oof_predictions_177.csv")
print(oof_df.head())
print("Number of OOF predictions:", len(oof_df))


Using device: cuda
GPU: NVIDIA A100-SXM4-40GB
Resolved image directory: images177
Number of image files: 177
Sample files: ['1.png', '10.png', '100.PNG', '101.PNG', '102.PNG', '104.PNG', '105.PNG', '106.PNG', '107.PNG', '108.PNG']

=== Dataset Check ===
Number of cases: 177
  filename label
0    1.png    1b
1    2.png     2
2    3.png     3
3    5.png    1b
4    6.png     3
Label mapping: {'1b': 0, '1c': 1, '2': 2, '3': 3}
All image files listed in dataset_177.csv were found.

===== Fold 1/5 =====
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 243MB/s]


[Ep 1] Train Loss 1.5150 | mAUC 0.4904 | bAUC 0.5489 || Val Loss 1.6893 | mAUC 0.5396 | bAUC 0.4611
[Ep 2] Train Loss 1.1979 | mAUC 0.7069 | bAUC 0.7758 || Val Loss 1.6565 | mAUC 0.5763 | bAUC 0.5611
[Ep 3] Train Loss 0.8816 | mAUC 0.8569 | bAUC 0.8904 || Val Loss 1.6516 | mAUC 0.5766 | bAUC 0.6556
[Ep 4] Train Loss 0.6971 | mAUC 0.9325 | bAUC 0.9587 || Val Loss 1.7495 | mAUC 0.5771 | bAUC 0.6667
[Ep 5] Train Loss 0.5135 | mAUC 0.9692 | bAUC 0.9771 || Val Loss 1.8612 | mAUC 0.5545 | bAUC 0.7611
[Ep 6] Train Loss 0.4309 | mAUC 0.9744 | bAUC 0.9962 || Val Loss 2.0149 | mAUC 0.5296 | bAUC 0.7611
[Ep 7] Train Loss 0.2810 | mAUC 0.9890 | bAUC 0.9992 || Val Loss 2.1647 | mAUC 0.5285 | bAUC 0.7667
[Ep 8] Train Loss 0.2478 | mAUC 0.9915 | bAUC 0.9985 || Val Loss 2.2805 | mAUC 0.5646 | bAUC 0.7389
[Ep 9] Train Loss 0.1737 | mAUC 0.9958 | bAUC 1.0000 || Val Loss 2.4191 | mAUC 0.5463 | bAUC 0.7278
[Ep 10] Train Loss 0.1387 | mAUC 0.9995 | bAUC 1.0000 || Val Loss 2.5288 | mAUC 0.5212 | bAUC 0.7444

In [12]:


# ===================== CELL 1 =====================
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Reanalysis v3 for JBHI submission-ready 177-case analysis
=========================================================

Colab use:
1. Upload MRI_ALLdata_OOF177.xlsx to /content.
2. Optional: upload ESCC_3_177.xlsx to /content for inter-rater agreement.
3. Run this script in one Colab cell.

Major updates vs code5_7 and reanalysis v2:
- Uses cross-fitted isotonic calibration for calibrated OOF predictions.
- Uses RAW OOF predictions for discrimination metrics:
    AUC, ROC curves, Youden threshold, sensitivity, specificity, F1,
    paired AUC comparisons, and SHAP model ranking.
- Uses CROSS-FITTED CALIBRATED OOF predictions for calibration metrics:
    Brier score, calibration curves, calibration slope/intercept.
- Replaces the previous approximate "DeLong-type" test with paired permutation testing.
- Saves case-level OOF predictions for auditability.
- Outputs main performance, incremental value, temporal validation, Spearman correlation,
  SHAP tables/figures, PDP, and optional inter-rater agreement tables.

Important reporting note:
If this v3 script is used for the manuscript, Methods should state:
"Pairwise AUC comparisons were performed using paired permutation testing with
Bonferroni correction for multiple comparisons." Do not call these p-values DeLong tests.
"""

# =============================================================================
# 0. Install packages if needed
# =============================================================================
import sys
import subprocess


def ensure_package(pkg_name, import_name=None):
    import_name = import_name or pkg_name
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg_name])


ensure_package("lightgbm")
ensure_package("shap")
ensure_package("openpyxl")

# =============================================================================
# 1. Imports and global settings
# =============================================================================
import warnings
from pathlib import Path
from collections import Counter
from datetime import datetime

import numpy as np
import pandas as pd
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt
import matplotlib as mpl

from scipy.stats import spearmanr
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    brier_score_loss,
    confusion_matrix,
    precision_recall_fscore_support,
    cohen_kappa_score,
)
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore", category=UserWarning)

# Input files
INPUT_XLSX = Path("/content/MRI_ALLdata_OOF177.xlsx")
RATER_XLSX = Path("/content/ESCC_3_177.xlsx")  # optional

# Output folders
OUT_DIR = Path("/content/RESULTS_FIGURES_177_REANALYSIS_V3")
TABLE_OUT_DIR = Path("/content/RESULTS_TABLES_177_REANALYSIS_V3")
OUT_DIR.mkdir(parents=True, exist_ok=True)
TABLE_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Reproducibility/settings
RANDOM_STATE = 42
N_SPLITS = 5
N_BOOT = 2000
N_PERM = 5000

# Figure style
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 20,
    "axes.titlesize": 24,
    "axes.labelsize": 22,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 15,
    "figure.titlesize": 24,
    "axes.linewidth": 1.2,
    "lines.linewidth": 2.2,
    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,
    "xtick.major.size": 4,
    "ytick.major.size": 4,
    "savefig.dpi": 600,
    "axes.unicode_minus": False,
})

# =============================================================================
# 2. Utility mappings
# =============================================================================
def map_sex(x):
    s = str(x).strip().lower()
    if s in ["1", "m", "male", "男", "男性"]:
        return 1
    if s in ["0", "f", "female", "女", "女性"]:
        return 0
    return np.nan


def map_escc(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        x = str(int(x)) if float(x).is_integer() else str(x)
    s = str(x).strip().lower().replace(" ", "")
    return {"1a": 1, "1b": 2, "1c": 3, "2": 4, "3": 5}.get(s, np.nan)


def frankel_bin(x):
    s = str(x).strip().upper()
    if s in ["A", "B", "C"]:
        return 0
    if s in ["D", "E"]:
        return 1
    return np.nan


def map_yesno(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        if x in [1, 2]:
            return 1
        if x == 0:
            return 0
    s = str(x).strip().lower()
    if s in ["yes", "y", "true", "1", "2", "あり", "有"]:
        return 1
    if s in ["no", "n", "false", "0", "なし", "無"]:
        return 0
    return np.nan

# =============================================================================
# 3. Statistics
# =============================================================================
def safe_n_splits(y, requested=N_SPLITS):
    """Return a valid stratified split number for small or imbalanced samples."""
    y = np.asarray(y).astype(int)
    counts = np.bincount(y)
    counts = counts[counts > 0]
    if len(counts) < 2:
        raise ValueError("Need both outcome classes for stratified CV.")
    return int(min(requested, counts.min()))


def auc_ci_bootstrap(y, p, n_boot=N_BOOT, seed=RANDOM_STATE):
    """Bootstrap CI for AUC. Used with raw predictions for discrimination."""
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    rng = np.random.default_rng(seed)
    idx = np.arange(len(y))
    aucs = []
    for _ in range(n_boot):
        b = rng.choice(idx, len(idx), replace=True)
        if len(np.unique(y[b])) < 2:
            continue
        aucs.append(roc_auc_score(y[b], p[b]))
    auc = roc_auc_score(y, p)
    lo, hi = np.percentile(aucs, [2.5, 97.5])
    return auc, lo, hi


def delta_auc_ci_bootstrap(y, p1, p2, n_boot=N_BOOT, seed=RANDOM_STATE):
    """Paired bootstrap CI for AUC(p1)-AUC(p2). Use raw predictions."""
    y = np.asarray(y).astype(int)
    p1 = np.asarray(p1).astype(float)
    p2 = np.asarray(p2).astype(float)
    rng = np.random.default_rng(seed)
    idx = np.arange(len(y))
    deltas = []
    for _ in range(n_boot):
        b = rng.choice(idx, len(idx), replace=True)
        if len(np.unique(y[b])) < 2:
            continue
        deltas.append(roc_auc_score(y[b], p1[b]) - roc_auc_score(y[b], p2[b]))
    delta = roc_auc_score(y, p1) - roc_auc_score(y, p2)
    lo, hi = np.percentile(deltas, [2.5, 97.5])
    return delta, lo, hi


def paired_permutation_auc_test(y, p1, p2, n_perm=N_PERM, seed=RANDOM_STATE):
    """
    Paired permutation test for the difference in AUCs.
    Null: predictions p1 and p2 are exchangeable within each patient.
    Use raw predictions. Returns AUC(p1)-AUC(p2) and a two-sided p value.
    """
    y = np.asarray(y).astype(int)
    p1 = np.asarray(p1).astype(float)
    p2 = np.asarray(p2).astype(float)
    if len(np.unique(y)) < 2:
        return np.nan, np.nan

    observed = roc_auc_score(y, p1) - roc_auc_score(y, p2)
    rng = np.random.default_rng(seed)
    count = 0

    for _ in range(n_perm):
        swap = rng.random(len(y)) < 0.5
        pp1 = np.where(swap, p2, p1)
        pp2 = np.where(swap, p1, p2)
        delta_perm = roc_auc_score(y, pp1) - roc_auc_score(y, pp2)
        if abs(delta_perm) >= abs(observed):
            count += 1

    p_value = (count + 1) / (n_perm + 1)
    return observed, p_value


def get_binary_metrics(y, p, thr=None):
    """Binary metrics from raw predictions by default."""
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    if thr is None:
        fpr, tpr, thresholds = roc_curve(y, p)
        thr = thresholds[np.argmax(tpr - fpr)]
    yhat = (p >= thr).astype(int)

    cm = confusion_matrix(y, yhat, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    _, _, f1, _ = precision_recall_fscore_support(
        y, yhat, average="binary", zero_division=0
    )
    return sens, spec, f1, thr


def calibration_slope_intercept(y_true, p_pred_calibrated):
    """Calibration slope/intercept from calibrated predictions."""
    y_true = np.asarray(y_true).astype(int)
    p_pred = np.asarray(p_pred_calibrated).astype(float)
    eps = 1e-6
    p_pred = np.clip(p_pred, eps, 1 - eps)
    logit_p = np.log(p_pred / (1 - p_pred)).reshape(-1, 1)
    lr = LogisticRegression(fit_intercept=True, solver="lbfgs")
    lr.fit(logit_p, y_true)
    return float(lr.coef_[0][0]), float(lr.intercept_[0])


# ===================== CELL 2 =====================
# =============================================================================
# 4. Models with cross-fitted calibration
# =============================================================================
def lgb_params():
    return dict(
        objective="binary",
        metric="auc",
        learning_rate=0.05,
        num_leaves=31,
        n_estimators=500,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        verbosity=-1,
    )


def inner_oof_predictions_for_calibration(X_train, y_train, params, n_splits=N_SPLITS):
    """
    Create inner OOF predictions within the outer-training data.
    These inner OOF predictions are used only to fit the calibrator.
    """
    X_train = X_train.reset_index(drop=True)
    y_train = np.asarray(y_train).astype(int)
    n_inner = safe_n_splits(y_train, n_splits)
    skf_inner = StratifiedKFold(n_splits=n_inner, shuffle=True, random_state=RANDOM_STATE)
    inner_oof = np.zeros(len(y_train), dtype=float)

    for tr_i, va_i in skf_inner.split(X_train, y_train):
        m = lgb.LGBMClassifier(**params)
        m.fit(X_train.iloc[tr_i], y_train[tr_i])
        inner_oof[va_i] = m.predict_proba(X_train.iloc[va_i])[:, 1]

    return inner_oof


def run_lgb_oof_crossfit_calibrated(X, y, n_splits=N_SPLITS):
    """
    Outer OOF prediction with calibration fitted only on the outer-training set.

    For each outer fold:
      1. Build inner OOF predictions inside the outer-training set.
      2. Fit isotonic calibration on those inner OOF predictions.
      3. Train model on the full outer-training set.
      4. Predict raw probability for the held-out outer-test fold.
      5. Apply the outer-training calibrator to the held-out outer-test prediction.

    Returns:
      p_raw: raw OOF predictions used for discrimination.
      p_cal: cross-fitted calibrated OOF predictions used for calibration/Brier.
      fold:  outer fold ID.
      final_model: model trained on all data, used for SHAP/PDP only.
    """
    X = X.reset_index(drop=True)
    y = np.asarray(y).astype(int)
    params = lgb_params()
    n_outer = safe_n_splits(y, n_splits)
    skf_outer = StratifiedKFold(n_splits=n_outer, shuffle=True, random_state=RANDOM_STATE)

    p_raw = np.zeros(len(y), dtype=float)
    p_cal = np.zeros(len(y), dtype=float)
    fold_id = np.zeros(len(y), dtype=int)

    for fold, (tr, te) in enumerate(skf_outer.split(X, y), start=1):
        X_tr = X.iloc[tr].reset_index(drop=True)
        y_tr = y[tr]
        X_te = X.iloc[te]

        inner_oof = inner_oof_predictions_for_calibration(X_tr, y_tr, params, n_splits=n_splits)
        iso = IsotonicRegression(out_of_bounds="clip")
        iso.fit(inner_oof, y_tr)

        model = lgb.LGBMClassifier(**params)
        model.fit(X_tr, y_tr)

        raw_te = model.predict_proba(X_te)[:, 1]
        cal_te = iso.transform(raw_te)

        p_raw[te] = raw_te
        p_cal[te] = cal_te
        fold_id[te] = fold

    final_model = lgb.LGBMClassifier(**params)
    final_model.fit(X, y)

    return {
        "p_raw": p_raw,
        "p_cal": p_cal,
        "fold": fold_id,
        "final_model": final_model,
    }


def run_lgb_temporal_train_test_calibrated(X_train, y_train, X_test):
    """
    Temporal validation.
    Raw test predictions are used for discrimination.
    Cross-fitted calibration inside training data is used for Brier/calibration.
    """
    X_train = X_train.reset_index(drop=True)
    X_test = X_test.reset_index(drop=True)
    y_train = np.asarray(y_train).astype(int)
    params = lgb_params()

    inner_oof = inner_oof_predictions_for_calibration(X_train, y_train, params, n_splits=N_SPLITS)
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(inner_oof, y_train)

    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)

    p_test_raw = model.predict_proba(X_test)[:, 1]
    p_test_cal = iso.transform(p_test_raw)

    return p_test_raw, p_test_cal, model, iso


# ===================== CELL 3 =====================
# =============================================================================
# 5. Plotting functions
# =============================================================================
def plot_roc(ax, y, curves, title):
    """
    curves: list of (label, raw probabilities, linestyle).
    ROC/AUC should use raw predictions.
    """
    for label, p, linestyle in curves:
        fpr, tpr, _ = roc_curve(y, p)
        auc_val = roc_auc_score(y, p)
        ax.plot(
            fpr, tpr,
            color="black",
            lw=2.2,
            linestyle=linestyle,
            label=f"{label}\n(AUC={auc_val:.3f})"
        )

    ax.plot([0, 1], [0, 1], "--", color="gray", lw=1.2)
    ax.set_title(title, fontsize=24)
    ax.set_xlabel("1 - Specificity", fontsize=22)
    ax.set_ylabel("Sensitivity", fontsize=22)
    ax.tick_params(labelsize=20)
    ax.legend(loc="lower right", frameon=True, fontsize=14)


def plot_calibration(ax, y, p_calibrated):
    """Calibration curve should use cross-fitted calibrated predictions."""
    d = pd.DataFrame({"y": y, "p": p_calibrated})
    try:
        d["bin"] = pd.qcut(d["p"], q=10, duplicates="drop")
        g = d.groupby("bin", observed=False).agg(
            obs=("y", "mean"),
            pred=("p", "mean")
        ).reset_index()
        ax.plot(g["pred"], g["obs"], "o-", color="black", lw=2, markersize=5)
    except Exception:
        ax.plot([np.mean(p_calibrated)], [np.mean(y)], "o", color="black", markersize=5)

    ax.plot([0, 1], [0, 1], "--", color="gray", lw=1.2)
    ax.set_title("Calibration plot", fontsize=24)
    ax.set_xlabel("Predicted probability", fontsize=22)
    ax.set_ylabel("Observed frequency", fontsize=22)
    ax.tick_params(labelsize=20)


def shap_summary_plot(model, X, title, filename, rename_map=None):
    X_disp = X.rename(columns=rename_map) if rename_map else X.copy()
    explainer = shap.TreeExplainer(model)
    sv = explainer.shap_values(X)

    if isinstance(sv, list):
        sv = sv[1]
    elif isinstance(sv, np.ndarray) and sv.ndim == 3:
        # shape may be (n_samples, n_features, n_classes)
        sv = sv[:, :, 1]

    shap.summary_plot(sv, X_disp, show=False)

    fig = plt.gcf()
    ax = plt.gca()
    ax.tick_params(axis="x", labelsize=14)
    for t in ax.get_yticklabels():
        t.set_fontsize(14)

    try:
        cbar = fig.axes[-1]
        cbar.tick_params(labelsize=14)
        cbar.set_ylabel(cbar.get_ylabel(), fontsize=14)
    except Exception:
        pass

    plt.title(title, fontsize=20)
    plt.tight_layout()
    plt.savefig(OUT_DIR / filename, dpi=600, bbox_inches="tight")
    plt.close()


def shap_mean_heatmap(models, X_dict, features, filename, rename_map=None):
    display_features = [rename_map.get(f, f) if rename_map else f for f in features]
    table = pd.DataFrame(index=display_features, columns=models.keys())

    for tag, model in models.items():
        X = X_dict[tag]
        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(X)

        if isinstance(sv, list):
            sv = sv[1]
        elif isinstance(sv, np.ndarray) and sv.ndim == 3:
            sv = sv[:, :, 1]

        abs_mean = np.abs(sv).mean(axis=0)

        for f_orig, f_disp, val in zip(features, display_features, abs_mean):
            table.loc[f_disp, tag] = float(val)

    if "3M" in table.columns:
        table = table.sort_values(by="3M", ascending=False)

    table_num = table.astype(float)

    fig = plt.figure(figsize=(6.8, 9.5))
    ax = fig.add_axes([0.50, 0.06, 0.24, 0.90])

    im = ax.imshow(
        table_num.values,
        cmap="Greys",
        aspect="auto",
        vmin=0.0,
        vmax=2.5
    )

    ax.set_xticks(np.arange(len(table_num.columns)))
    ax.set_xticklabels(table_num.columns, fontsize=16)
    ax.set_yticks(np.arange(len(table_num.index)))
    ax.set_yticklabels(table_num.index, fontsize=16)

    threshold = 1.25
    for i in range(table_num.shape[0]):
        for j in range(table_num.shape[1]):
            val = float(table_num.iloc[i, j])
            color = "white" if val > threshold else "black"
            ax.text(j, i, f"{val:.3f}", ha="center", va="center", color=color, fontsize=10)

    cax = fig.add_axes([0.75, 0.06, 0.022, 0.90])
    cbar = fig.colorbar(im, cax=cax, ticks=np.arange(0.0, 2.5 + 0.001, 0.5))
    cbar.set_label("Mean(|SHAP value|)", fontsize=16)
    cbar.ax.tick_params(labelsize=16)

    fig.savefig(OUT_DIR / filename, dpi=600, bbox_inches="tight")
    plt.close(fig)

    return table_num


def compute_and_plot_pdp(models_by_horizon, X_dict, out_dir):
    out_dir = Path(out_dir)
    required_tags = ["3M", "6M", "12M"]

    for tag in required_tags:
        if tag not in models_by_horizon:
            raise KeyError(f"Missing model for horizon: {tag}")
        if tag not in X_dict:
            raise KeyError(f"Missing predictor matrix for horizon: {tag}")
        if "ESCC" not in X_dict[tag].columns:
            raise KeyError(f"'ESCC' column not found in X_dict['{tag}']")

    escc_levels = sorted(X_dict["3M"]["ESCC"].dropna().unique())
    pdp = {}

    for tag in required_tags:
        model = models_by_horizon[tag]
        X_base = X_dict[tag].copy()
        vals = []

        for g in escc_levels:
            X_temp = X_base.copy()
            X_temp["ESCC"] = g
            prob_alive = model.predict_proba(X_temp)[:, 1]
            vals.append(np.mean(1.0 - prob_alive))

        pdp[tag] = np.array(vals)

    label_map = {1: "1a", 2: "1b", 3: "1c", 4: "2", 5: "3"}
    tick_labels = [label_map.get(int(x), str(x)) for x in escc_levels]

    plt.figure(figsize=(8, 6))
    plt.plot(escc_levels, pdp["3M"], color="black", lw=2.8, linestyle="-", marker="o", label="3M mortality")
    plt.plot(escc_levels, pdp["6M"], color="black", lw=2.4, linestyle="--", marker="o", label="6M mortality")
    plt.plot(escc_levels, pdp["12M"], color="black", lw=2.4, linestyle=":", marker="o", label="12M mortality")
    plt.xlabel("ESCC grade")
    plt.ylabel("Mean predicted mortality risk")
    plt.xticks(escc_levels, tick_labels)
    plt.legend(frameon=True, loc="center left", bbox_to_anchor=(1.02, 0.5), borderpad=0.8)
    plt.tight_layout()

    out_path = out_dir / "Supplementary_ESCC_PDP_all_bw_v3.png"
    plt.savefig(out_path, dpi=600, bbox_inches="tight")
    plt.close()

    return pd.DataFrame({
        "ESCC_numeric": escc_levels,
        "ESCC_label": tick_labels,
        "PDP_3M_mortality": pdp["3M"],
        "PDP_6M_mortality": pdp["6M"],
        "PDP_12M_mortality": pdp["12M"],
    })


# ===================== CELL 4 =====================
# =============================================================================
# 6. Optional inter-rater agreement
# =============================================================================
def fleiss_kappa(M):
    M = np.asarray(M, dtype=float)
    N, k = M.shape
    n = M.sum(axis=1)[0]
    p_j = M.sum(axis=0) / (N * n)
    P_i = ((M * (M - 1)).sum(axis=1)) / (n * (n - 1))
    P_bar = P_i.mean()
    P_e = (p_j ** 2).sum()
    kappa = (P_bar - P_e) / (1 - P_e)
    return kappa, P_bar, P_e


def quadratic_weighted_kappa(a, b, labels_order):
    lab_to_i = {lab: i for i, lab in enumerate(labels_order)}
    a_idx = np.array([lab_to_i[str(x).strip()] for x in a])
    b_idx = np.array([lab_to_i[str(x).strip()] for x in b])
    n_cat = len(labels_order)

    w = np.zeros((n_cat, n_cat))
    for i in range(n_cat):
        for j in range(n_cat):
            w[i, j] = ((i - j) ** 2) / ((n_cat - 1) ** 2)

    O = np.zeros((n_cat, n_cat))
    for x, y in zip(a_idx, b_idx):
        O[x, y] += 1
    O = O / O.sum()

    a_hist = np.bincount(a_idx, minlength=n_cat)
    b_hist = np.bincount(b_idx, minlength=n_cat)
    E = np.outer(a_hist, b_hist)
    E = E / E.sum()

    return 1.0 - (w * O).sum() / (w * E).sum()


def consensus_row(row, raters, label_to_idx, idx_to_label):
    vals = [str(row[x]).strip() for x in raters]
    counts = Counter(vals)
    most_common_label, freq = counts.most_common(1)[0]

    if freq >= 2:
        return most_common_label

    idxs = sorted(label_to_idx[v] for v in vals)
    return idx_to_label[idxs[1]]


def run_inter_rater_agreement_if_available():
    if not RATER_XLSX.exists():
        print(f"[SKIP] Optional rater file not found: {RATER_XLSX}")
        return None

    labels = ["1a", "1b", "1c", "2", "3"]
    label_to_idx = {lab: i for i, lab in enumerate(labels)}
    idx_to_label = {i: lab for lab, i in label_to_idx.items()}
    raters = ["ESCC_1", "ESCC_2", "ESCC_3"]

    rater_df = pd.read_excel(RATER_XLSX)
    missing = [c for c in raters if c not in rater_df.columns]
    if missing:
        raise ValueError(f"Missing rater columns in {RATER_XLSX}: {missing}")

    for c in raters:
        rater_df[c] = rater_df[c].astype(str).str.strip()

    M = np.zeros((len(rater_df), len(labels)), dtype=int)
    for i, row in rater_df[raters].iterrows():
        for val in row:
            if val not in label_to_idx:
                raise ValueError(f"Unexpected ESCC label: {val}")
            M[i, label_to_idx[val]] += 1

    fleiss_k, P_bar, P_e = fleiss_kappa(M)

    def landis_koch_label(kappa):
        if kappa < 0:
            return "poor"
        if kappa <= 0.20:
            return "slight"
        if kappa <= 0.40:
            return "fair"
        if kappa <= 0.60:
            return "moderate"
        if kappa <= 0.80:
            return "substantial"
        return "almost perfect"

    fleiss_df = pd.DataFrame([{
        "n": len(rater_df),
        "Fleiss_kappa": fleiss_k,
        "P_bar_mean_observed_agreement": P_bar,
        "P_e_chance_agreement": P_e,
        "Interpretation_Landis_Koch": landis_koch_label(fleiss_k),
    }])

    pair_records = []
    pairs = [("ESCC_1", "ESCC_2"), ("ESCC_1", "ESCC_3"), ("ESCC_2", "ESCC_3")]

    for c1, c2 in pairs:
        k_unw = cohen_kappa_score(rater_df[c1], rater_df[c2])
        k_w = quadratic_weighted_kappa(rater_df[c1], rater_df[c2], labels)
        pair_records.append({
            "Rater_pair": f"{c1} vs {c2}",
            "Unweighted_Cohen_kappa": k_unw,
            "Unweighted_interpretation_Landis_Koch": landis_koch_label(k_unw),
            "Quadratic_weighted_Cohen_kappa": k_w,
            "Weighted_interpretation_Landis_Koch": landis_koch_label(k_w),
        })

    pair_df = pd.DataFrame(pair_records)

    rater_df["ESCC_consensus"] = rater_df.apply(
        consensus_row,
        axis=1,
        args=(raters, label_to_idx, idx_to_label),
    )

    dist_df = rater_df["ESCC_consensus"].value_counts().sort_index().reset_index()
    dist_df.columns = ["ESCC_consensus", "n"]

    full_disagree = (
        (rater_df["ESCC_1"] != rater_df["ESCC_2"]) &
        (rater_df["ESCC_1"] != rater_df["ESCC_3"]) &
        (rater_df["ESCC_2"] != rater_df["ESCC_3"])
    ).sum()

    full_disagree_df = pd.DataFrame([{"full_disagreement_cases": int(full_disagree)}])

    out_path = TABLE_OUT_DIR / "01_inter_rater_agreement_177_v3.xlsx"
    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        fleiss_df.to_excel(writer, sheet_name="Fleiss_kappa", index=False)
        pair_df.to_excel(writer, sheet_name="Pairwise_Cohen_kappa", index=False)
        dist_df.to_excel(writer, sheet_name="Consensus_distribution", index=False)
        full_disagree_df.to_excel(writer, sheet_name="Full_disagreement", index=False)
        rater_df.to_excel(writer, sheet_name="Ratings_with_consensus", index=False)

    print(f"Saved inter-rater agreement results: {out_path}")
    return fleiss_df, pair_df

# =============================================================================
# 7. Load and prepare analysis dataset
# =============================================================================
if not INPUT_XLSX.exists():
    raise FileNotFoundError(
        f"Input file not found: {INPUT_XLSX}\n"
        "Please upload MRI_ALLdata_OOF177.xlsx to the Colab /content folder."
    )

df_raw = pd.read_excel(INPUT_XLSX)
df_raw = df_raw.loc[:, ~df_raw.columns.str.contains("^Unnamed")].copy()
df_raw.columns = df_raw.columns.str.strip()

required_cols = [
    "Age",
    "Sex",
    "Number of Spinal Metastases",
    "Serum Albumin",
    "CRP",
    "ESCC",
    "Expert",
    "Performance Status (ECOG)",
    "Frankel Grade",
    "Barthel Index (ADL)",
    "Malignancy (Katagiri Score)",
    "Visceral Metastasis (Yes=1/No=0)",
    "Body Mass Index (BMI)",
    "Revised Tokuhashi score",
    "New Katagiri score",
    "3-Month Survival (0=Death, 1=Alive)",
    "6-Month Survival (0=Death, 1=Alive)",
    "12-Month Survival (0=Death, 1=Alive)",
]

missing_cols = [c for c in required_cols if c not in df_raw.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in Excel: {missing_cols}")

id_candidates = ["patient_id", "Patient ID", "ID", "case_id", "Case ID", "filename", "Filename"]
id_col = next((c for c in id_candidates if c in df_raw.columns), None)
case_id = df_raw[id_col].astype(str).values if id_col else df_raw.index.astype(str).values

df = pd.DataFrame({
    "case_id": case_id,
    "row_index": df_raw.index,
    "Age": pd.to_numeric(df_raw["Age"], errors="coerce"),
    "Sex": df_raw["Sex"].apply(map_sex),
    "Number of spinal metastases": pd.to_numeric(df_raw["Number of Spinal Metastases"], errors="coerce"),
    "Albumin": pd.to_numeric(df_raw["Serum Albumin"], errors="coerce"),
    "CRP": pd.to_numeric(df_raw["CRP"], errors="coerce"),
    "ESCC": df_raw["ESCC"].apply(map_escc),
    "ESCC_expert": df_raw["Expert"].apply(map_escc),
    "ECOG": pd.to_numeric(df_raw["Performance Status (ECOG)"], errors="coerce"),
    "Frankel grade": df_raw["Frankel Grade"].apply(frankel_bin),
    "Barthel Index": pd.to_numeric(df_raw["Barthel Index (ADL)"], errors="coerce"),
    "Malignancy (New Katagiri score)": pd.to_numeric(df_raw["Malignancy (Katagiri Score)"], errors="coerce"),
    "Visceral metastasis": df_raw["Visceral Metastasis (Yes=1/No=0)"].apply(map_yesno),
    "BMI": pd.to_numeric(df_raw["Body Mass Index (BMI)"], errors="coerce"),
    "Tokuhashi_binary": (pd.to_numeric(df_raw["Revised Tokuhashi score"], errors="coerce") >= 9).astype(float),
    "Katagiri_binary": (pd.to_numeric(df_raw["New Katagiri score"], errors="coerce") < 7).astype(float),
    "Y_3M": pd.to_numeric(df_raw["3-Month Survival (0=Death, 1=Alive)"], errors="coerce"),
    "Y_6M": pd.to_numeric(df_raw["6-Month Survival (0=Death, 1=Alive)"], errors="coerce"),
    "Y_12M": pd.to_numeric(df_raw["12-Month Survival (0=Death, 1=Alive)"], errors="coerce"),
})

DATE_CANDIDATES = ["ope date", "ope_date", "Operation Date", "operation date"]
date_col = next((c for c in DATE_CANDIDATES if c in df_raw.columns), None)

if date_col is not None:
    df["ope_date"] = pd.to_datetime(df_raw[date_col], errors="coerce")
else:
    df["ope_date"] = pd.NaT

missing_summary = df.isna().sum().reset_index()
missing_summary.columns = ["analysis_column", "missing_count"]
missing_summary.to_excel(TABLE_OUT_DIR / "00_missing_value_summary_177_v3.xlsx", index=False)

print("=== Missing values by analysis column ===")
print(missing_summary)

FEATURES_FULL_CNN = [
    "CRP",
    "Malignancy (New Katagiri score)",
    "Age",
    "Barthel Index",
    "Albumin",
    "BMI",
    "ESCC",
    "ECOG",
    "Number of spinal metastases",
    "Sex",
    "Frankel grade",
    "Visceral metastasis",
]

FEATURES_CLINICAL = [f for f in FEATURES_FULL_CNN if f != "ESCC"]
FEATURES_EXPERT = ["ESCC_expert" if f == "ESCC" else f for f in FEATURES_FULL_CNN]

FEATURE_RENAME = {
    "CRP": "CRP",
    "Malignancy (New Katagiri score)": "Malignancy (New Katagiri score)",
    "Age": "Age",
    "Barthel Index": "Barthel Index",
    "Albumin": "Albumin",
    "BMI": "BMI",
    "ESCC": "ESCC grade",
    "ESCC_expert": "Expert ESCC grade",
    "ECOG": "ECOG performance status",
    "Number of spinal metastases": "Number of spinal metastases",
    "Sex": "Sex",
    "Frankel grade": "Frankel grade",
    "Visceral metastasis": "Visceral metastasis",
}

# =============================================================================
# 8. Optional inter-rater agreement
# =============================================================================
run_inter_rater_agreement_if_available()

# =============================================================================
# 9. Spearman correlation
# =============================================================================
corr_sub = df.dropna(subset=["ESCC", "ECOG", "Barthel Index"]).copy()

rho_ecog, p_ecog = spearmanr(corr_sub["ESCC"], corr_sub["ECOG"])
rho_barthel, p_barthel = spearmanr(corr_sub["ESCC"], corr_sub["Barthel Index"])

spearman_df = pd.DataFrame([
    {
        "Comparison": "ESCC vs ECOG",
        "Spearman_rho": rho_ecog,
        "p_value": p_ecog,
        "n": len(corr_sub),
    },
    {
        "Comparison": "ESCC vs Barthel Index",
        "Spearman_rho": rho_barthel,
        "p_value": p_barthel,
        "n": len(corr_sub),
    },
])

spearman_df.to_excel(TABLE_OUT_DIR / "07_spearman_correlation_177_v3.xlsx", index=False)

print("\n=== Spearman correlation ===")
print(spearman_df)


# ===================== CELL 5 =====================
# =============================================================================
# 10. Main full-model performance vs conventional scores
# =============================================================================
main_records = []
case_prediction_records = []
models_by_horizon = {}
X_dict = {}

for tag, ycol in [("3M", "Y_3M"), ("6M", "Y_6M"), ("12M", "Y_12M")]:
    required = FEATURES_FULL_CNN + [ycol, "Tokuhashi_binary", "Katagiri_binary"]
    sub = df.dropna(subset=required).copy().reset_index(drop=True)
    X = sub[FEATURES_FULL_CNN]
    y = sub[ycol].astype(int).values

    print(f"\n=== Main full-model analysis: {tag} ===")
    print(f"n = {len(sub)}")

    res_full = run_lgb_oof_crossfit_calibrated(X, y)

    # v3 distinction:
    # raw predictions for discrimination; calibrated predictions for Brier/calibration.
    p_full_raw = res_full["p_raw"]
    p_full_cal = res_full["p_cal"]
    fold = res_full["fold"]

    models_by_horizon[tag] = res_full["final_model"]
    X_dict[tag] = X

    p_tok = sub["Tokuhashi_binary"].astype(float).values
    p_kat = sub["Katagiri_binary"].astype(float).values

    full_auc, full_lo, full_hi = auc_ci_bootstrap(y, p_full_raw)
    tok_auc, tok_lo, tok_hi = auc_ci_bootstrap(y, p_tok)
    kat_auc, kat_lo, kat_hi = auc_ci_bootstrap(y, p_kat)

    d_tok, d_tok_lo, d_tok_hi = delta_auc_ci_bootstrap(y, p_full_raw, p_tok)
    d_kat, d_kat_lo, d_kat_hi = delta_auc_ci_bootstrap(y, p_full_raw, p_kat)

    _, p_full_vs_tok = paired_permutation_auc_test(
        y, p_full_raw, p_tok, seed=RANDOM_STATE + 11
    )
    _, p_full_vs_kat = paired_permutation_auc_test(
        y, p_full_raw, p_kat, seed=RANDOM_STATE + 12
    )

    sens, spec, f1, thr = get_binary_metrics(y, p_full_raw, thr=None)

    brier_cal = brier_score_loss(y, p_full_cal)
    brier_raw = brier_score_loss(y, p_full_raw)
    slope_cal, intercept_cal = calibration_slope_intercept(y, p_full_cal)

    main_records.append({
        "Horizon": tag,
        "n": len(sub),

        "Full_AUC_raw": full_auc,
        "Full_AUC_raw_95CI_low": full_lo,
        "Full_AUC_raw_95CI_high": full_hi,

        "Full_Brier_calibrated": brier_cal,
        "Full_Brier_raw": brier_raw,
        "Full_calibration_slope_calibrated": slope_cal,
        "Full_calibration_intercept_calibrated": intercept_cal,

        "Full_sensitivity_Youden_raw": sens,
        "Full_specificity_Youden_raw": spec,
        "Full_F1_Youden_raw": f1,
        "Full_Youden_threshold_raw": thr,

        "Tokuhashi_AUC": tok_auc,
        "Tokuhashi_AUC_95CI_low": tok_lo,
        "Tokuhashi_AUC_95CI_high": tok_hi,

        "New_Katagiri_AUC": kat_auc,
        "New_Katagiri_AUC_95CI_low": kat_lo,
        "New_Katagiri_AUC_95CI_high": kat_hi,

        "Delta_AUC_raw_Full_minus_Tokuhashi": d_tok,
        "Delta_AUC_raw_Full_minus_Tokuhashi_95CI_low": d_tok_lo,
        "Delta_AUC_raw_Full_minus_Tokuhashi_95CI_high": d_tok_hi,
        "p_paired_permutation_raw_Full_vs_Tokuhashi": p_full_vs_tok,

        "Delta_AUC_raw_Full_minus_New_Katagiri": d_kat,
        "Delta_AUC_raw_Full_minus_New_Katagiri_95CI_low": d_kat_lo,
        "Delta_AUC_raw_Full_minus_New_Katagiri_95CI_high": d_kat_hi,
        "p_paired_permutation_raw_Full_vs_New_Katagiri": p_full_vs_kat,
    })

    for i in range(len(sub)):
        case_prediction_records.append({
            "case_id": sub.loc[i, "case_id"],
            "row_index": sub.loc[i, "row_index"],
            "Horizon": tag,
            "Outcome_alive": int(y[i]),
            "OOF_fold": int(fold[i]),
            "p_full_raw": float(p_full_raw[i]),
            "p_full_calibrated": float(p_full_cal[i]),
            "p_Tokuhashi_binary": float(p_tok[i]),
            "p_Katagiri_binary": float(p_kat[i]),
        })

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.8))
    plot_roc(
        axes[0],
        y,
        [
            ("Full model", p_full_raw, "-"),
            ("Revised Tokuhashi", p_tok, "--"),
            ("New Katagiri", p_kat, ":"),
        ],
        f"{tag} survival"
    )
    plot_calibration(axes[1], y, p_full_cal)
    plt.tight_layout()
    plt.savefig(
        OUT_DIR / f"Figure_{tag}_ROC_raw_Calibration_calibrated_reanalysis_v3.png",
        dpi=600,
        bbox_inches="tight",
    )
    plt.close()

    shap_summary_plot(
        res_full["final_model"],
        X,
        f"SHAP summary ({tag})",
        f"SHAP_{tag}_177_reanalysis_v3.png",
        rename_map=FEATURE_RENAME,
    )

main_df = pd.DataFrame(main_records)
main_df.to_excel(TABLE_OUT_DIR / "04_main_performance_table_177_v3.xlsx", index=False)

case_pred_df = pd.DataFrame(case_prediction_records)
case_pred_df.to_excel(TABLE_OUT_DIR / "03_survival_oof_predictions_177_v3.xlsx", index=False)

print("\n=== Main performance v3 ===")
print(main_df)

shap_table = shap_mean_heatmap(
    models_by_horizon,
    X_dict,
    FEATURES_FULL_CNN,
    "SHAP_heatmap_3M_6M_12M_reanalysis_v3.png",
    rename_map=FEATURE_RENAME,
)
shap_table.to_excel(TABLE_OUT_DIR / "08_shap_mean_absolute_values_177_v3.xlsx")

pdp_df = compute_and_plot_pdp(models_by_horizon, X_dict, OUT_DIR)
pdp_df.to_excel(TABLE_OUT_DIR / "09_escc_pdp_177_v3.xlsx", index=False)


# ===================== CELL 6 =====================
# =============================================================================
# 11. Incremental value analysis
# =============================================================================
incremental_rows = []
incremental_prediction_records = []

for tag, ycol in [("3M", "Y_3M"), ("6M", "Y_6M"), ("12M", "Y_12M")]:
    required = FEATURES_CLINICAL + ["ESCC_expert", "ESCC", ycol]
    sub = df.dropna(subset=required).copy().reset_index(drop=True)
    y = sub[ycol].astype(int).values

    X_clin = sub[FEATURES_CLINICAL]
    X_exp = sub[FEATURES_EXPERT]
    X_cnn = sub[FEATURES_FULL_CNN]

    print(f"\n=== Incremental value analysis v3: {tag} ===")
    print(f"n = {len(sub)}")

    res_clin = run_lgb_oof_crossfit_calibrated(X_clin, y)
    res_exp = run_lgb_oof_crossfit_calibrated(X_exp, y)
    res_cnn = run_lgb_oof_crossfit_calibrated(X_cnn, y)

    preds_raw = {
        "Clinical only": res_clin["p_raw"],
        "Clinical + Expert ESCC": res_exp["p_raw"],
        "Clinical + CNN ESCC": res_cnn["p_raw"],
    }

    preds_cal = {
        "Clinical only": res_clin["p_cal"],
        "Clinical + Expert ESCC": res_exp["p_cal"],
        "Clinical + CNN ESCC": res_cnn["p_cal"],
    }

    p_clin_raw = preds_raw["Clinical only"]
    p_clin_cal = preds_cal["Clinical only"]
    brier_clin_cal = brier_score_loss(y, p_clin_cal)

    for model_name in ["Clinical only", "Clinical + Expert ESCC", "Clinical + CNN ESCC"]:
        p_raw = preds_raw[model_name]
        p_cal = preds_cal[model_name]

        auc_val, auc_lo, auc_hi = auc_ci_bootstrap(y, p_raw)
        brier_val_cal = brier_score_loss(y, p_cal)
        brier_val_raw = brier_score_loss(y, p_raw)

        if model_name == "Clinical only":
            delta_auc = np.nan
            delta_auc_lo = np.nan
            delta_auc_hi = np.nan
            p_vs_clin = np.nan
            delta_brier_cal = np.nan
        else:
            delta_auc, delta_auc_lo, delta_auc_hi = delta_auc_ci_bootstrap(y, p_raw, p_clin_raw)
            _, p_vs_clin = paired_permutation_auc_test(
                y,
                p_raw,
                p_clin_raw,
                seed=RANDOM_STATE + 101,
            )
            delta_brier_cal = brier_val_cal - brier_clin_cal

        incremental_rows.append({
            "Horizon": tag,
            "n": len(sub),
            "Model": model_name,

            "AUC_raw": auc_val,
            "AUC_raw_95CI_low": auc_lo,
            "AUC_raw_95CI_high": auc_hi,

            "Brier_calibrated": brier_val_cal,
            "Brier_raw": brier_val_raw,

            "Delta_AUC_raw_vs_clinical_only": delta_auc,
            "Delta_AUC_raw_vs_clinical_only_95CI_low": delta_auc_lo,
            "Delta_AUC_raw_vs_clinical_only_95CI_high": delta_auc_hi,
            "p_paired_permutation_raw_vs_clinical_only": p_vs_clin,

            "Delta_Brier_calibrated_vs_clinical_only": delta_brier_cal,
        })

    delta_cnn_exp, delta_cnn_exp_lo, delta_cnn_exp_hi = delta_auc_ci_bootstrap(
        y,
        preds_raw["Clinical + CNN ESCC"],
        preds_raw["Clinical + Expert ESCC"],
    )
    _, p_cnn_vs_exp = paired_permutation_auc_test(
        y,
        preds_raw["Clinical + CNN ESCC"],
        preds_raw["Clinical + Expert ESCC"],
        seed=RANDOM_STATE + 202,
    )

    incremental_rows.append({
        "Horizon": tag,
        "n": len(sub),
        "Model": "CNN ESCC vs Expert ESCC comparison",

        "AUC_raw": np.nan,
        "AUC_raw_95CI_low": np.nan,
        "AUC_raw_95CI_high": np.nan,

        "Brier_calibrated": np.nan,
        "Brier_raw": np.nan,

        "Delta_AUC_raw_vs_clinical_only": np.nan,
        "Delta_AUC_raw_vs_clinical_only_95CI_low": np.nan,
        "Delta_AUC_raw_vs_clinical_only_95CI_high": np.nan,
        "p_paired_permutation_raw_vs_clinical_only": np.nan,

        "Delta_Brier_calibrated_vs_clinical_only": np.nan,

        "Delta_AUC_raw_CNN_minus_Expert": delta_cnn_exp,
        "Delta_AUC_raw_CNN_minus_Expert_95CI_low": delta_cnn_exp_lo,
        "Delta_AUC_raw_CNN_minus_Expert_95CI_high": delta_cnn_exp_hi,
        "p_paired_permutation_raw_CNN_vs_Expert": p_cnn_vs_exp,
    })

    for i in range(len(sub)):
        incremental_prediction_records.append({
            "case_id": sub.loc[i, "case_id"],
            "row_index": sub.loc[i, "row_index"],
            "Horizon": tag,
            "Outcome_alive": int(y[i]),

            "p_clinical_only_raw": float(preds_raw["Clinical only"][i]),
            "p_clinical_plus_expert_ESCC_raw": float(preds_raw["Clinical + Expert ESCC"][i]),
            "p_clinical_plus_CNN_ESCC_raw": float(preds_raw["Clinical + CNN ESCC"][i]),

            "p_clinical_only_calibrated": float(preds_cal["Clinical only"][i]),
            "p_clinical_plus_expert_ESCC_calibrated": float(preds_cal["Clinical + Expert ESCC"][i]),
            "p_clinical_plus_CNN_ESCC_calibrated": float(preds_cal["Clinical + CNN ESCC"][i]),
        })

incremental_df = pd.DataFrame(incremental_rows)
incremental_df.to_excel(TABLE_OUT_DIR / "05_incremental_value_table_177_v3.xlsx", index=False)

pd.DataFrame(incremental_prediction_records).to_excel(
    TABLE_OUT_DIR / "03b_incremental_oof_predictions_177_v3.xlsx",
    index=False,
)

print("\n=== Incremental value analysis v3 ===")
print(incremental_df)


# ===================== CELL 7 =====================
# =============================================================================
# 12. Temporal validation
# =============================================================================
temporal_records = []
temporal_prediction_records = []
cutoff_date = pd.to_datetime("2018-12-31")

if df["ope_date"].isna().all():
    print("[SKIP] No operation-date column found. Temporal validation was not performed.")
else:
    for tag, ycol in [("3M", "Y_3M"), ("6M", "Y_6M"), ("12M", "Y_12M")]:
        required = FEATURES_FULL_CNN + [ycol, "ope_date"]
        sub = df.dropna(subset=required).copy().reset_index(drop=True)

        train = sub[sub["ope_date"] <= cutoff_date].copy().reset_index(drop=True)
        test = sub[sub["ope_date"] > cutoff_date].copy().reset_index(drop=True)

        print(f"\n=== Temporal validation v3: {tag} ===")
        print(f"Train n = {len(train)}")
        print(f"Test n  = {len(test)}")

        if len(train) == 0 or len(test) == 0:
            print("[SKIP] Empty train or test cohort.")
            continue

        y_train = train[ycol].astype(int).values
        y_test = test[ycol].astype(int).values

        if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
            print("[SKIP] Need both outcome classes in train and test.")
            continue

        X_train = train[FEATURES_FULL_CNN]
        X_test = test[FEATURES_FULL_CNN]

        p_test_raw, p_test_cal, model_temp, iso_temp = run_lgb_temporal_train_test_calibrated(
            X_train,
            y_train,
            X_test,
        )

        auc_temp, auc_temp_lo, auc_temp_hi = auc_ci_bootstrap(y_test, p_test_raw)
        brier_temp_cal = brier_score_loss(y_test, p_test_cal)
        brier_temp_raw = brier_score_loss(y_test, p_test_raw)
        slope_temp, intercept_temp = calibration_slope_intercept(y_test, p_test_cal)
        sens_temp, spec_temp, f1_temp, thr_temp = get_binary_metrics(y_test, p_test_raw, thr=None)

        temporal_records.append({
            "Horizon": tag,
            "cutoff_date": str(cutoff_date.date()),
            "train_period": "2009-2018 or <= 2018-12-31",
            "test_period": "2019-2023 or > 2018-12-31",

            "n_train": len(train),
            "n_test": len(test),

            "train_alive": int(np.sum(y_train)),
            "train_death": int(len(y_train) - np.sum(y_train)),
            "test_alive": int(np.sum(y_test)),
            "test_death": int(len(y_test) - np.sum(y_test)),

            "AUC_temporal_raw": auc_temp,
            "AUC_temporal_raw_95CI_low": auc_temp_lo,
            "AUC_temporal_raw_95CI_high": auc_temp_hi,

            "Brier_temporal_calibrated": brier_temp_cal,
            "Brier_temporal_raw": brier_temp_raw,

            "Sensitivity_Youden_raw": sens_temp,
            "Specificity_Youden_raw": spec_temp,
            "F1_Youden_raw": f1_temp,
            "Youden_threshold_raw": thr_temp,

            "Calibration_slope_calibrated": slope_temp,
            "Calibration_intercept_calibrated": intercept_temp,
        })

        for i in range(len(test)):
            temporal_prediction_records.append({
                "case_id": test.loc[i, "case_id"],
                "row_index": test.loc[i, "row_index"],
                "Horizon": tag,
                "Outcome_alive": int(y_test[i]),
                "p_temporal_raw": float(p_test_raw[i]),
                "p_temporal_calibrated": float(p_test_cal[i]),
            })

        fig, axes = plt.subplots(1, 2, figsize=(14, 5.8))
        plot_roc(
            axes[0],
            y_test,
            [("Temporal full model", p_test_raw, "-")],
            f"{tag} temporal validation"
        )
        plot_calibration(axes[1], y_test, p_test_cal)
        plt.tight_layout()
        plt.savefig(
            OUT_DIR / f"Figure_{tag}_Temporal_ROC_raw_Calibration_calibrated_reanalysis_v3.png",
            dpi=600,
            bbox_inches="tight",
        )
        plt.close()

    temporal_df = pd.DataFrame(temporal_records)
    temporal_df.to_excel(TABLE_OUT_DIR / "06_temporal_validation_177_v3.xlsx", index=False)

    pd.DataFrame(temporal_prediction_records).to_excel(
        TABLE_OUT_DIR / "06b_temporal_validation_predictions_177_v3.xlsx",
        index=False,
    )

    print("\n=== Temporal validation v3 ===")
    print(temporal_df)

# =============================================================================
# 13. Reanalysis log
# =============================================================================
log_lines = [
    "Reanalysis v3 completed.",
    f"Timestamp: {datetime.now().isoformat(timespec='seconds')}",
    f"Input file: {INPUT_XLSX}",
    f"Optional rater file present: {RATER_XLSX.exists()}",
    f"Random state: {RANDOM_STATE}",
    f"Outer CV splits: {N_SPLITS}",
    f"Bootstrap iterations: {N_BOOT}",
    f"Permutation iterations: {N_PERM}",
    "Calibration: cross-fitted isotonic calibration within outer CV.",
    "Discrimination metrics: raw OOF predictions.",
    "Calibration metrics: cross-fitted calibrated OOF predictions.",
    "AUC comparisons: paired permutation testing, not DeLong.",
    "Full model features: " + ", ".join(FEATURES_FULL_CNN),
    "Clinical-only features: " + ", ".join(FEATURES_CLINICAL),
    "Temporal validation cutoff: 2018-12-31.",
    f"Figure output folder: {OUT_DIR}",
    f"Table output folder: {TABLE_OUT_DIR}",
]

(TABLE_OUT_DIR / "08_reanalysis_log_v3.txt").write_text("\n".join(log_lines), encoding="utf-8")

print("\nAll reanalysis v3 outputs saved.")
print("Figure outputs:", OUT_DIR)
print("Table outputs :", TABLE_OUT_DIR)


=== Missing values by analysis column ===
                    analysis_column  missing_count
0                           case_id              0
1                         row_index              0
2                               Age              0
3                               Sex              0
4       Number of spinal metastases              0
5                           Albumin              0
6                               CRP              0
7                              ESCC              0
8                       ESCC_expert              0
9                              ECOG              0
10                    Frankel grade              0
11                    Barthel Index              0
12  Malignancy (New Katagiri score)              0
13              Visceral metastasis              0
14                              BMI              0
15                 Tokuhashi_binary              0
16                  Katagiri_binary              0
17                             Y_3M     


=== Main full-model analysis: 6M ===
n = 177



=== Main full-model analysis: 12M ===
n = 177



=== Main performance v3 ===
  Horizon    n  Full_AUC_raw  Full_AUC_raw_95CI_low  Full_AUC_raw_95CI_high  \
0      3M  177      0.710949               0.623693                0.797335   
1      6M  177      0.795838               0.727867                0.858360   
2     12M  177      0.826104               0.762193                0.884206   

   Full_Brier_calibrated  Full_Brier_raw  Full_calibration_slope_calibrated  \
0               0.188622        0.230880                           0.035001   
1               0.190122        0.214255                           0.420941   
2               0.173780        0.191193                           0.194105   

   Full_calibration_intercept_calibrated  Full_sensitivity_Youden_raw  ...  \
0                               1.176772                     0.598540  ...   
1                              -0.040644                     0.778846  ...   
2                              -0.306413                     0.753247  ...   

   New_Katagiri_AUC_95CI


=== Incremental value analysis v3: 3M ===
n = 177

=== Incremental value analysis v3: 6M ===
n = 177

=== Incremental value analysis v3: 12M ===
n = 177

=== Incremental value analysis v3 ===
   Horizon    n                               Model   AUC_raw  \
0       3M  177                       Clinical only  0.726095   
1       3M  177              Clinical + Expert ESCC  0.719343   
2       3M  177                 Clinical + CNN ESCC  0.710949   
3       3M  177  CNN ESCC vs Expert ESCC comparison       NaN   
4       6M  177                       Clinical only  0.798867   
5       6M  177              Clinical + Expert ESCC  0.803082   
6       6M  177                 Clinical + CNN ESCC  0.795838   
7       6M  177  CNN ESCC vs Expert ESCC comparison       NaN   
8      12M  177                       Clinical only  0.832338   
9      12M  177              Clinical + Expert ESCC  0.847922   
10     12M  177                 Clinical + CNN ESCC  0.826104   
11     12M  177  CNN ESCC v


=== Temporal validation v3: 6M ===
Train n = 129
Test n  = 48



=== Temporal validation v3: 12M ===
Train n = 129
Test n  = 48



=== Temporal validation v3 ===
  Horizon cutoff_date                train_period                test_period  \
0      3M  2018-12-31  2009-2018 or <= 2018-12-31  2019-2023 or > 2018-12-31   
1      6M  2018-12-31  2009-2018 or <= 2018-12-31  2019-2023 or > 2018-12-31   
2     12M  2018-12-31  2009-2018 or <= 2018-12-31  2019-2023 or > 2018-12-31   

   n_train  n_test  train_alive  train_death  test_alive  test_death  ...  \
0      129      48           97           32          40           8  ...   
1      129      48           67           62          37          11  ...   
2      129      48           50           79          27          21  ...   

   AUC_temporal_raw_95CI_low  AUC_temporal_raw_95CI_high  \
0                   0.449398                    0.875149   
1                   0.699070                    0.930582   
2                   0.721415                    0.947559   

   Brier_temporal_calibrated  Brier_temporal_raw  Sensitivity_Youden_raw  \
0                   0